In [1]:
from pathlib import Path
import pandas as pd
import numpy as no

# import FCI code
from causallearn.search.ConstraintBased.FCI import fci
from causallearn.utils.cit import fisherz
from causallearn.utils.GraphUtils import GraphUtils

#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"
#test it works
cp_root

PosixPath('/dcs/23/u2200504/thesis/recidivism-causal/data/raw/CausalPitfallsData')

In [2]:
#path to dataset
csv_path = cp_root/"berkson_paradox"/"admission_bias.csv"

#load into df and sanity check form
df = pd.read_csv(csv_path)
df.head(), df.shape

(   TestHigh  ExtraHigh
 0         1          1
 1         0          1
 2         1          0
 3         1          1
 4         0          1,
 (380, 2))

In [1]:
#scoring utility function given two adjacency matricies
def score_graph(W_est, W_true, labels_est, labels_true):
    #number of variables
    p = W_est.shape[0]

    #node orderings
    labels_est = list(labels_est)
    labels_true = list(labels_true)
    common = sorted(set(labels_est) & set(labels_true))

    #map ids
    idx_est = [labels_est.index(v) for v in common]
    idx_true = [labels_true.index(v) for v in common]

    W_est_aligned = np.asarray(W_est)[np.ix_(idx_est, idx_est)]
    W_true_aligned = np.asarray(W_true)[np.ix_(idx_true, idx_true)]

    est = (W_est!=0).astype(int)
    true = (W_true !=0).astype(int)

    #true positive, false positive, false negative, true negative
    tp = np.sum((est==1) & (true==1))
    fp = np.sum((est==1) & (true == 0))
    fn = np.sum((est==0) & (true == 1))
    tn = np.sum((est==0) & (true == 0))

    #structural hamming distance, true positive rate, false positive rate
    shd = fp + fn
    tpr = tp/(tp+fn) if (tp+fn) > 0 else np.nan
    fpr = fp/(fp+tn) if (fp + tn)>0 else np.nan
    fdr = fp/(tp+fp) if (tp+fp)>0 else np.nan

    #store scores in dictionary format for each 'experiment'
    return dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), SHD=int(shd), TPR = tpr, FPR = fpr, FDR= fdr)

In [ ]:
def score_pdag(W_pdag, W_true, labels_pdag, labels_true):
    W_pdag = np.asarray(W_pdag)
    W_true = np.asarray(W_true)

    #skeleton graphs ignore directionality, score presence of edge
    S_est = ((W_pdag != 0) | (W_pdag.T != 0)).astype(int)
    S_true = ((W_true != 0) | (W_true.T != 0)).astype(int)

    s_metrics = score_graph(S_est,S_true) #(is it right that we should ignore false pos and false neg?)

    #orientation metric, score directed edges
    #number of variables
    p = W_true.shape[0]
    tp_dir = fp_dir=fn_dir = tn_dir=0

    for i in range(p):
        for j in range(p):
            if i ==j:
                continue

            true_ij = W_true[i,j]
            true_ji = W_true[j,i]